# Task 3.1：資料轉換為 Llama-Factory SFT 訓練格式

## 目標

將 Task 2 的 NSL-KDD 網路連線資料，轉換為 **LlamaFactory SFT（監督微調）** 所需的訓練資料格式，讓微調後的 LLM 能根據網路連線特徵，自動輸出結構化的**資安事件分析 JSON 報告**。

## 輸出格式：ShareGPT

採用 **ShareGPT 格式**（`conversations` 陣列），因為 Llama 3.2 是 Chat 模型，ShareGPT 直接映射 `user`/`assistant` 角色，不需要框架額外轉換。

## 刻意製造資料不平衡

為了在 Task 3.3/3.4 觀察「資料量對微調效果的影響」，**刻意維持不平衡的類別比例**：

| 類別 | 抽樣筆數 | 佔比 |
|------|---------|------|
| Normal | 500 | 44.2% |
| DoS | 500 | 44.2% |
| Probe | 100 | 8.8% |
| R2L | 20 | 1.8% |
| U2R | 10 | 0.9% |
| **合計** | **1,130** | 100% |

預期結果：微調後模型對 DoS（500 筆）能產出高品質報告，但對 U2R（10 筆）可能出現格式崩潰或幻覺。

---

## JSON Schema 設計

每筆訓練資料的「模型預期輸出」是一個包含 **7 個欄位**的 JSON 物件：

| 欄位 | 型別 | 說明 | 範例 |
|------|------|------|------|
| `event_type` | string | 攻擊類別 | `"dos"`, `"u2r"`, `"normal"` |
| `severity` | string | 嚴重等級 | `"critical"`, `"high"`, `"medium"`, `"low"`, `"info"` |
| `source_ip` | string | 來源 IP（模擬） | `"203.45.67.89"` |
| `destination_ip` | string | 目的 IP（模擬） | `"10.0.0.5"` |
| `protocol` | string | 通訊協定 | `"tcp"`, `"udp"`, `"icmp"` |
| `description` | string | 事件描述 | 自然語言描述連線行為與異常原因 |
| `recommendation` | string | 處置建議 | 給 SOC 分析師的具體行動指引 |

### ShareGPT 格式範例

```json
{
  "conversations": [
    {
      "from": "human",
      "value": "你是一個資安事件分析助手。請根據以下網路連線特徵，輸出 JSON 格式的資安事件分析報告。\n報告必須包含：event_type、severity、source_ip、destination_ip、protocol、description、recommendation。\n\nduration=0, protocol_type=tcp, service=private, flag=S0, src_bytes=0, dst_bytes=0, ..."
    },
    {
      "from": "gpt",
      "value": "{\n  \"event_type\": \"dos\",\n  \"severity\": \"high\",\n  ...\n}"
    }
  ]
}
```

> **備註**：NSL-KDD 原始資料不含 IP 位址，`source_ip` 和 `destination_ip` 會依攻擊類型模擬生成合理的 IP 範圍。

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

RANDOM_STATE = 42
DATA_PATH = Path('data/KDD/kdd_nsl_like.parquet')

df = pd.read_parquet(DATA_PATH)
print(f'資料集大小：{df.shape[0]:,} 筆 × {df.shape[1]} 欄')
print(f'\n五大攻擊類別分佈：')
print(df['attack_type'].value_counts())
print(f'\n細分攻擊名稱（target）分佈：')
print(df['target'].value_counts())

資料集大小：145,585 筆 × 43 欄

五大攻擊類別分佈：
attack_type
normal    87831
dos       54572
probe      2131
r2l         999
u2r          52
Name: count, dtype: int64

細分攻擊名稱（target）分佈：
target
normal.             87831
neptune.            51820
back.                 968
teardrop.             918
satan.                906
warezclient.          893
ipsweep.              651
smurf.                641
portsweep.            416
pod.                  206
nmap.                 158
guess_passwd.          53
buffer_overflow.       30
warezmaster.           20
land.                  19
imap.                  12
rootkit.               10
loadmodule.             9
ftp_write.              8
multihop.               7
phf.                    4
perl.                   3
spy.                    2
Name: count, dtype: int64


---

## 分層抽樣（Stratified Sampling）

從 145,585 筆資料中，依作業要求的不平衡比例抽樣：

- **Normal / DoS**：各 500 筆（資料充足類別）
- **Probe**：100 筆
- **R2L**：20 筆
- **U2R**：10 筆（資料集中僅有 52 筆）

使用固定隨機種子（`seed=42`）確保結果可重現。

In [2]:
sample_counts = {
    'normal': 500,
    'dos':    500,
    'probe':  100,
    'r2l':     20,
    'u2r':     10,
}

rng = np.random.default_rng(RANDOM_STATE)
sampled_rows = []

for cat, n in sample_counts.items():
    pool = df[df['attack_type'] == cat]
    idx = rng.choice(len(pool), size=n, replace=False)
    sampled_rows.append(pool.iloc[idx])

df_sft = pd.concat(sampled_rows).reset_index(drop=True)

print(f'抽樣後總筆數：{len(df_sft):,}')
print(f'\n各類別筆數：')
print(df_sft['attack_type'].value_counts())
print(f'\n各 target 細分分佈：')
for cat in ['normal', 'dos', 'probe', 'r2l', 'u2r']:
    subset = df_sft[df_sft['attack_type'] == cat]
    targets = subset['target'].value_counts()
    print(f'\n【{cat.upper()}】（{len(subset)} 筆）')
    for t, c in targets.items():
        print(f'  {t} → {c} 筆')

抽樣後總筆數：1,130

各類別筆數：
attack_type
normal    500
dos       500
probe     100
r2l        20
u2r        10
Name: count, dtype: int64

各 target 細分分佈：

【NORMAL】（500 筆）
  normal. → 500 筆

【DOS】（500 筆）
  neptune. → 476 筆
  back. → 11 筆
  teardrop. → 7 筆
  smurf. → 3 筆
  pod. → 3 筆

【PROBE】（100 筆）
  satan. → 43 筆
  ipsweep. → 30 筆
  portsweep. → 21 筆
  nmap. → 6 筆

【R2L】（20 筆）
  warezclient. → 18 筆
  guess_passwd. → 2 筆

【U2R】（10 筆）
  buffer_overflow. → 4 筆
  rootkit. → 3 筆
  loadmodule. → 2 筆
  perl. → 1 筆


---

## IP 位址模擬 & 嚴重等級對應

### IP 模擬策略

NSL-KDD 不含 IP 位址，需依攻擊類型模擬合理的來源/目的 IP：

| 攻擊類型 | 來源 IP | 目的 IP | 理由 |
|---------|--------|--------|------|
| Normal | 內網 192.168.1.x | 外網隨機 | 合法對外連線 |
| DoS | 外網隨機 | 內網伺服器 10.0.0.x | 外部攻擊內部服務 |
| Probe | 外網隨機 | 內網子網段 10.0.x.x | 掃描內部網段 |
| R2L | 外網隨機 | 內網 192.168.1.x | 遠端嘗試入侵 |
| U2R | 內網 192.168.1.x | 內網 192.168.1.x | 本地提權（已在內網） |

### 嚴重等級對應

| 攻擊類別 | severity | 理由 |
|---------|----------|------|
| Normal | info | 正常流量 |
| Probe | medium | 偵察階段，尚未造成損害 |
| DoS | high | 影響服務可用性 |
| R2L | high | 可能已取得未授權存取 |
| U2R | critical | 已提升至 root 權限，最高風險 |

In [3]:
def generate_ip(attack_type, rng):
    """依攻擊類型產生合理的模擬 IP"""
    if attack_type == 'normal':
        src = f"192.168.1.{rng.integers(1, 255)}"
        dst = f"{rng.integers(1, 224)}.{rng.integers(0, 256)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}"
    elif attack_type == 'dos':
        src = f"{rng.integers(1, 224)}.{rng.integers(0, 256)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}"
        dst = f"10.0.0.{rng.integers(1, 21)}"
    elif attack_type == 'probe':
        src = f"{rng.integers(1, 224)}.{rng.integers(0, 256)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}"
        dst = f"10.0.{rng.integers(0, 6)}.{rng.integers(1, 255)}"
    elif attack_type == 'r2l':
        src = f"{rng.integers(1, 224)}.{rng.integers(0, 256)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}"
        dst = f"192.168.1.{rng.integers(1, 51)}"
    else:  # u2r
        src = f"192.168.1.{rng.integers(1, 255)}"
        dst = f"192.168.1.{rng.integers(1, 11)}"
    return src, dst


SEVERITY_MAP = {
    'normal': 'info',
    'dos':    'high',
    'probe':  'medium',
    'r2l':    'high',
    'u2r':    'critical',
}

print('IP 生成函式與 Severity 對應表定義完成')
# 測試
test_rng = np.random.default_rng(0)
for at in ['normal', 'dos', 'probe', 'r2l', 'u2r']:
    src, dst = generate_ip(at, test_rng)
    print(f'  {at:8s} → src={src:18s}  dst={dst:18s}  severity={SEVERITY_MAP[at]}')

IP 生成函式與 Severity 對應表定義完成
  normal   → src=192.168.1.217       dst=143.130.69.79       severity=info
  dos      → src=10.19.4.45          dst=10.0.0.17           severity=high
  probe    → src=145.233.128.155     dst=10.0.5.186          severity=medium
  r2l      → src=141.139.143.238     dst=192.168.1.14        severity=high
  u2r      → src=192.168.1.208       dst=192.168.1.7         severity=critical


---

## Description & Recommendation 生成

### 品質設計原則

1. **依 `target`（細分攻擊名稱）做分支**：每種攻擊子類型（如 neptune、smurf、buffer_overflow）有不同描述模板
2. **插入實際特徵值**：在模板中嵌入 `src_bytes`、`flag`、`count`、`num_failed_logins` 等數值，讓同子類型的不同筆資料也有差異
3. **Recommendation 多樣化**：每類攻擊提供 5 條不同建議，依 `hash((target, idx))` 選擇，避免大量重複

### 涵蓋的 23 種 target

| 攻擊大類 | target 名稱 |
|---------|------------|
| Normal | normal |
| DoS | neptune, smurf, back, teardrop, pod, land |
| Probe | satan, ipsweep, portsweep, nmap |
| R2L | warezclient, guess_passwd, warezmaster, ftp_write, imap, multihop, phf, spy |
| U2R | buffer_overflow, rootkit, loadmodule, perl |

In [4]:
def generate_description(row):
    """根據連線特徵產生自然語言描述"""
    attack = row['attack_type']
    target_name = row['target'].rstrip('.')
    proto = row['protocol_type']
    service = row['service']
    flag = row['flag']
    src_bytes = row['src_bytes']
    dst_bytes = row['dst_bytes']
    duration = row['duration']
    count = row['count']
    srv_count = row['srv_count']
    dst_host_count = row['dst_host_count']
    serror_rate = row['serror_rate']
    same_srv_rate = row['same_srv_rate']
    diff_srv_rate = row['diff_srv_rate']
    dst_host_serror_rate = row['dst_host_serror_rate']
    dst_host_srv_serror_rate = row['dst_host_srv_serror_rate']
    dst_host_same_src_port_rate = row['dst_host_same_src_port_rate']

    if attack == 'normal':
        return (
            f"一筆正常的 {proto.upper()} 連線，目標服務為 {service}，"
            f"連線旗標 {flag}，傳送 {src_bytes:,} bytes、接收 {dst_bytes:,} bytes，"
            f"持續 {duration} 秒。過去 2 秒內同主機連線數={count}，同服務連線數={srv_count}，"
            f"目的主機連線數={dst_host_count}，同服務比率={same_srv_rate}。未偵測到異常行為。"
        )

    elif attack == 'dos':
        templates = {
            'neptune': (
                f"SYN Flood 攻擊（neptune）：大量 {proto.upper()} SYN 封包湧入目標的 {service} 服務"
                f"（flag={flag}），未完成三向交握，消耗伺服器連線資源。"
                f"過去 2 秒內同主機連線數={count}，SYN 錯誤率={serror_rate}，"
                f"目的主機連線數={dst_host_count}，不同服務比率={diff_srv_rate}，"
                f"目的主機 SYN 錯誤率={dst_host_srv_serror_rate}。"
            ),
            'smurf': (
                f"Smurf 放大攻擊：利用 {proto.upper()} 廣播封包放大流量，"
                f"來源封包 {src_bytes:,} bytes，目標接收 {dst_bytes:,} bytes，"
                f"同服務連線數={srv_count}，目的主機連線數={dst_host_count}，"
                f"同來源端口比率={dst_host_same_src_port_rate}，企圖以放大流量癱瘓目標主機。"
            ),
            'back': (
                f"Back 攻擊：對 {service} 服務發送大量惡意請求"
                f"（傳送 {src_bytes:,} bytes、接收 {dst_bytes:,} bytes），"
                f"持續 {duration} 秒，同主機連線數={count}，"
                f"目的主機連線數={dst_host_count}，試圖耗盡伺服器處理能力。"
            ),
            'teardrop': (
                f"Teardrop 攻擊：透過 {proto.upper()} 發送重疊的 IP 分片封包"
                f"（{src_bytes:,} bytes），flag={flag}，同主機連線數={count}，"
                f"目的主機連線數={dst_host_count}，企圖使目標系統在重組封包時崩潰。"
            ),
            'pod': (
                f"Ping of Death 攻擊：發送超大 {proto.upper()} 封包"
                f"（{src_bytes:,} bytes），超出目標緩衝區承受範圍，"
                f"同主機連線數={count}，目的主機連線數={dst_host_count}，"
                f"可能導致系統崩潰或重新啟動。"
            ),
            'land': (
                f"Land 攻擊：{proto.upper()} 封包的來源與目的位址相同，"
                f"企圖使目標系統陷入自我連線迴圈，"
                f"flag={flag}，服務={service}，同主機連線數={count}。"
            ),
        }
        return templates.get(target_name,
            f"DoS 攻擊（{target_name}）：透過 {proto.upper()}/{service} "
            f"發動阻斷服務，flag={flag}，傳送 {src_bytes:,} bytes、接收 {dst_bytes:,} bytes，"
            f"同主機連線數={count}，目的主機連線數={dst_host_count}。"
        )

    elif attack == 'probe':
        templates = {
            'ipsweep': (
                f"IP 掃描（ipsweep）：對目標子網段發送 {proto.upper()} 探測封包，"
                f"嘗試發現存活主機，過去 2 秒內發起 {count} 次連線，"
                f"flag={flag}，目的主機連線數={dst_host_count}，"
                f"同來源端口比率={dst_host_same_src_port_rate}，目的主機錯誤率={dst_host_serror_rate}。"
            ),
            'portsweep': (
                f"端口掃描（portsweep）：對目標主機的 {service} 等多個服務端口進行掃描，"
                f"使用 {proto.upper()} 協定，flag={flag}，"
                f"同服務連線數={srv_count}，目的主機連線數={dst_host_count}，"
                f"同來源端口比率={dst_host_same_src_port_rate}，嘗試找出開放端口。"
            ),
            'satan': (
                f"SATAN 弱點掃描：系統化探測目標主機的已知弱點，"
                f"使用 {proto.upper()} 協定存取 {service} 服務，"
                f"flag={flag}，連線次數={count}，不同服務比率={diff_srv_rate}，"
                f"目的主機錯誤率={dst_host_serror_rate}。"
            ),
            'nmap': (
                f"Nmap 掃描：利用 {proto.upper()} flag={flag} 進行作業系統指紋辨識與服務偵測，"
                f"目標服務={service}，傳送 {src_bytes:,} bytes、接收 {dst_bytes:,} bytes，"
                f"同主機連線數={count}，目的主機連線數={dst_host_count}。"
            ),
        }
        return templates.get(target_name,
            f"偵察掃描（{target_name}）：使用 {proto.upper()} 協定探測 {service} 服務，"
            f"flag={flag}，count={count}，目的主機連線數={dst_host_count}，"
            f"目的主機錯誤率={dst_host_serror_rate}。"
        )

    elif attack == 'r2l':
        templates = {
            'guess_passwd': (
                f"密碼猜測攻擊（guess_passwd）：對 {service} 服務進行暴力密碼猜測，"
                f"失敗登入次數={row['num_failed_logins']}，"
                f"傳送 {src_bytes:,} bytes、接收 {dst_bytes:,} bytes，"
                f"同主機連線數={count}，目的主機連線數={dst_host_count}。"
            ),
            'warezclient': (
                f"非法軟體下載（warezclient）：透過 {service} 服務下載未授權檔案，"
                f"已成功登入（logged_in={row['logged_in']}），"
                f"傳輸量 {dst_bytes:,} bytes，持續 {duration} 秒，"
                f"同主機連線數={count}，目的主機連線數={dst_host_count}。"
            ),
            'warezmaster': (
                f"非法軟體散佈（warezmaster）：利用 {service} 服務散佈盜版軟體，"
                f"傳送 {src_bytes:,} bytes，持續 {duration} 秒，"
                f"flag={flag}，目的主機連線數={dst_host_count}。"
            ),
            'ftp_write': (
                f"FTP 寫入攻擊（ftp_write）：利用 FTP 服務上傳惡意檔案，"
                f"檔案建立數={row['num_file_creations']}，"
                f"傳送 {src_bytes:,} bytes，持續 {duration} 秒。"
            ),
            'imap': (
                f"IMAP 攻擊：利用 IMAP 郵件協定漏洞嘗試取得遠端存取權限，"
                f"flag={flag}，傳送 {src_bytes:,} bytes、接收 {dst_bytes:,} bytes，"
                f"持續 {duration} 秒。"
            ),
            'multihop': (
                f"多跳攻擊（multihop）：攻擊者透過多個跳板主機連接，"
                f"存取 {service} 服務，logged_in={row['logged_in']}，"
                f"持續 {duration} 秒，傳送 {src_bytes:,} bytes，企圖隱藏真實來源。"
            ),
            'phf': (
                f"PHF 攻擊：利用 CGI 程式 phf 的漏洞，"
                f"透過 {service} 服務執行任意指令，"
                f"傳送 {src_bytes:,} bytes、接收 {dst_bytes:,} bytes，持續 {duration} 秒。"
            ),
            'spy': (
                f"間諜程式（spy）：在取得 {service} 服務存取後植入間諜程式，"
                f"logged_in={row['logged_in']}，持續 {duration} 秒，"
                f"傳送 {src_bytes:,} bytes，竊取敏感資料。"
            ),
        }
        return templates.get(target_name,
            f"R2L 攻擊（{target_name}）：從遠端對 {service} 服務嘗試取得本地權限，"
            f"flag={flag}，傳送 {src_bytes:,} bytes，持續 {duration} 秒。"
        )

    else:  # u2r
        templates = {
            'buffer_overflow': (
                f"緩衝區溢位攻擊（buffer_overflow）：利用程式漏洞寫入 {src_bytes:,} bytes 惡意資料，"
                f"root_shell={row['root_shell']}，su_attempted={row['su_attempted']}，"
                f"透過 {service} 服務提升至 root 權限，持續 {duration} 秒。"
            ),
            'rootkit': (
                f"Rootkit 植入（rootkit）：在取得本地存取後安裝 rootkit，"
                f"透過 {service} 服務操作，root_shell={row['root_shell']}，"
                f"num_root={row['num_root']}，持續 {duration} 秒，企圖持續控制系統。"
            ),
            'loadmodule': (
                f"Loadmodule 攻擊：利用核心模組載入漏洞，"
                f"透過 {service} 服務提升權限，"
                f"root_shell={row['root_shell']}，傳送 {src_bytes:,} bytes，持續 {duration} 秒。"
            ),
            'perl': (
                f"Perl 攻擊：利用 Perl 直譯器漏洞，"
                f"透過 {service} 服務嘗試提升至 root 權限，"
                f"root_shell={row['root_shell']}，持續 {duration} 秒，傳送 {src_bytes:,} bytes。"
            ),
        }
        return templates.get(target_name,
            f"U2R 提權攻擊（{target_name}）：本地使用者嘗試提升至 root 權限，"
            f"root_shell={row['root_shell']}，service={service}，持續 {duration} 秒。"
        )


print('generate_description() 定義完成')
# 測試
test_row = df_sft[df_sft['attack_type'] == 'dos'].iloc[0]
print(f'\n測試（{test_row["target"].rstrip(".")}）：')
print(generate_description(test_row))

generate_description() 定義完成

測試（neptune）：
SYN Flood 攻擊（neptune）：大量 TCP SYN 封包湧入目標的 private 服務（flag=REJ），未完成三向交握，消耗伺服器連線資源。過去 2 秒內同主機連線數=293，SYN 錯誤率=0.0，目的主機連線數=255，不同服務比率=0.05，目的主機 SYN 錯誤率=0.0。


In [5]:
def generate_recommendation(attack_type, target_name, row, idx):
    """產生處置建議"""
    service = row['service']

    if attack_type == 'normal':
        return "無需處置，此為正常網路連線。持續監控即可。"

    recs = {
        'dos': [
            f"立即啟用 SYN Cookie 防護，限制來源 IP 對 {service} 服務的連線速率，並通知網路團隊監控流量。",
            f"在防火牆上暫時封鎖攻擊來源 IP，並聯繫 ISP 上游進行流量清洗（DDoS mitigation）。",
            f"檢查 {service} 服務的最大連線數設定，必要時啟動備援伺服器分散負載。",
            f"啟用入侵防禦系統（IPS）的 DoS 防護規則，自動丟棄異常封包，同時保留日誌供事後分析。",
            f"立即將受攻擊的 {service} 服務移至 DDoS 防護代理（如 Cloudflare）後方，減輕直接衝擊。",
        ],
        'probe': [
            f"將來源 IP 加入監控名單，觀察是否有後續攻擊行為。同時檢查 {service} 服務是否有已知弱點。",
            f"檢查被掃描的端口與服務，關閉不必要的對外服務，減少攻擊面。",
            f"部署 IDS 規則偵測同來源的連續掃描行為，設定警報閾值以即時通知分析師。",
            f"檢視防火牆規則，確認僅開放必要的 {service} 服務端口，封鎖未授權的偵察封包。",
            f"對目標主機進行弱點掃描，確認被探測的服務是否已修補所有已知漏洞。",
        ],
        'r2l': [
            f"立即重設 {service} 服務的所有使用者密碼，啟用多因素驗證（MFA）。",
            f"檢查 {service} 服務的存取日誌，確認是否有帳號已遭入侵，並封鎖可疑來源 IP。",
            f"封鎖來源 IP 並審查受影響主機上的檔案完整性，確認未被植入後門。",
            f"限制 {service} 服務的登入嘗試次數，啟用帳號鎖定機制，防止暴力破解。",
            f"檢視 {service} 服務的權限設定，確認最小權限原則，移除不必要的遠端存取帳號。",
        ],
        'u2r': [
            f"立即隔離受影響主機，中斷其網路連線。檢查 /etc/passwd 與 /etc/shadow 是否遭竄改。",
            f"進行完整的系統鑑識（forensics），檢查是否有 rootkit 或後門程式，保全所有日誌證據。",
            f"重新安裝作業系統並從已知安全的備份還原，所有憑證（包括 {service} 服務帳號）視為已洩漏並全數更換。",
            f"檢查 {service} 服務是否存在已知的緩衝區溢位漏洞，立即套用安全修補程式。",
            f"啟動事件應變程序（Incident Response），通知資安團隊與管理層，評估攻擊範圍是否擴及其他主機。",
        ],
    }

    options = recs.get(attack_type, ["進行人工調查並回報資安團隊。"])
    selection = hash((target_name, idx)) % len(options)
    return options[selection]


print('generate_recommendation() 定義完成')
# 測試
for i in range(3):
    rec = generate_recommendation('dos', 'neptune', df_sft.iloc[0], i)
    print(f'  neptune idx={i}: {rec[:60]}...')

generate_recommendation() 定義完成
  neptune idx=0: 立即啟用 SYN Cookie 防護，限制來源 IP 對 domain_u 服務的連線速率，並通知網路團隊監控流量。...
  neptune idx=1: 檢查 domain_u 服務的最大連線數設定，必要時啟動備援伺服器分散負載。...
  neptune idx=2: 在防火牆上暫時封鎖攻擊來源 IP，並聯繫 ISP 上游進行流量清洗（DDoS mitigation）。...


---

## 主轉換迴圈：組裝 ShareGPT 訓練資料

### 輸入特徵選擇

從 41 個特徵中挑選 **15 個關鍵特徵**組成模型的輸入，理由：

- 全部 41 個特徵中，許多欄位幾乎全為 0（如 `num_outbound_cmds`），對模型學習無幫助
- 過長的輸入會浪費模型的 context window
- 15 個特徵已涵蓋所有攻擊類型的判別信號

### 選定的 15 個特徵

| 特徵 | 重要性 |
|------|--------|
| `duration` | 連線持續時間 |
| `protocol_type` | 通訊協定（tcp/udp/icmp） |
| `service` | 目標服務（http/ftp/telnet 等） |
| `flag` | 連線狀態旗標（SF/S0/REJ 等） |
| `src_bytes` | 來源傳送位元組數 |
| `dst_bytes` | 目的接收位元組數 |
| `count` | 過去 2 秒內同主機連線數 |
| `srv_count` | 過去 2 秒內同服務連線數 |
| `serror_rate` | SYN 錯誤率 |
| `same_srv_rate` | 同服務連線比率 |
| `dst_host_count` | 目的主機連線數 |
| `dst_host_same_srv_rate` | 目的主機同服務連線比率 |
| `logged_in` | 是否成功登入 |
| `root_shell` | 是否取得 root shell |
| `num_failed_logins` | 失敗登入次數 |

In [6]:
KEY_FEATURES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes',
    'count', 'srv_count', 'serror_rate', 'same_srv_rate',
    'dst_host_count', 'dst_host_same_srv_rate',
    'logged_in', 'root_shell', 'num_failed_logins',
]

SYSTEM_INSTRUCTION = (
    "你是一個資安事件分析助手。請根據以下網路連線特徵，"
    "輸出 JSON 格式的資安事件分析報告。\n"
    "報告必須包含以下 7 個欄位：event_type、severity、source_ip、"
    "destination_ip、protocol、description、recommendation。"
)

rng = np.random.default_rng(RANDOM_STATE)
sft_data = []

for idx, row in df_sft.iterrows():
    attack = row['attack_type']
    target_name = row['target'].rstrip('.')
    src_ip, dst_ip = generate_ip(attack, rng)

    # 組裝模型預期輸出的 JSON
    output_json = {
        'event_type':      attack,
        'severity':        SEVERITY_MAP[attack],
        'source_ip':       src_ip,
        'destination_ip':  dst_ip,
        'protocol':        row['protocol_type'],
        'description':     generate_description(row),
        'recommendation':  generate_recommendation(attack, target_name, row, idx),
    }

    # 組裝模型的「輸入」：15 個關鍵特徵
    feature_parts = [f"{feat}={row[feat]}" for feat in KEY_FEATURES]
    feature_str = ', '.join(feature_parts)

    # ShareGPT 格式
    record = {
        'conversations': [
            {
                'from': 'human',
                'value': f"{SYSTEM_INSTRUCTION}\n\n{feature_str}"
            },
            {
                'from': 'gpt',
                'value': json.dumps(output_json, ensure_ascii=False, indent=2)
            }
        ]
    }
    sft_data.append(record)

# 打亂順序（避免同類別集中，影響訓練）
rng.shuffle(sft_data)

# 儲存
OUTPUT_PATH = Path('cybersecurity_sft_data.json')
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(sft_data, f, ensure_ascii=False, indent=2)

file_size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024
print(f'已產出 {len(sft_data)} 筆 SFT 訓練資料')
print(f'檔案：{OUTPUT_PATH}')
print(f'大小：{file_size_mb:.2f} MB')

已產出 1130 筆 SFT 訓練資料
檔案：cybersecurity_sft_data.json
大小：1.33 MB


---

## 品質驗證

對產出的 `cybersecurity_sft_data.json` 執行四項品質檢查：

1. **JSON 合法性**：每筆 gpt output 都能被 `json.loads()` 成功解析
2. **Schema 完整性**：每筆都包含 7 個必要欄位
3. **Description 唯一率**：> 80%（確保描述有足夠多樣性）
4. **類別分佈**：確認 normal=500、dos=500、probe=100、r2l=20、u2r=10

In [7]:
# 重新載入並驗證
with open('cybersecurity_sft_data.json', encoding='utf-8') as f:
    data = json.load(f)

print(f'總筆數：{len(data)}')

# --- 檢查 1：JSON 合法性 + Schema 完整性 ---
required_keys = ['event_type', 'severity', 'source_ip', 'destination_ip',
                 'protocol', 'description', 'recommendation']
errors = 0
for i, item in enumerate(data):
    output_str = item['conversations'][1]['value']
    try:
        parsed = json.loads(output_str)
        missing = [k for k in required_keys if k not in parsed]
        if missing:
            print(f'  第 {i} 筆缺少欄位：{missing}')
            errors += 1
    except json.JSONDecodeError as e:
        print(f'  第 {i} 筆 JSON 解析失敗：{e}')
        errors += 1

print(f'\n✅ JSON 合法性 + Schema 完整性：{len(data) - errors}/{len(data)} 通過（{errors} 筆有問題）')

總筆數：1130

✅ JSON 合法性 + Schema 完整性：1130/1130 通過（0 筆有問題）


In [8]:
# --- 檢查 2：Description 唯一率 ---
descriptions = []
for item in data:
    parsed = json.loads(item['conversations'][1]['value'])
    descriptions.append(parsed['description'])

unique_count = len(set(descriptions))
unique_ratio = unique_count / len(descriptions)
print(f'Description 唯一數：{unique_count} / {len(descriptions)}')
print(f'Description 唯一率：{unique_ratio:.1%}')
if unique_ratio >= 0.80:
    print('✅ 通過（>= 80%）')
else:
    print('⚠️ 未通過（< 80%），需增加描述多樣性')

# --- 檢查 3：Recommendation 唯一率 ---
recommendations = []
for item in data:
    parsed = json.loads(item['conversations'][1]['value'])
    recommendations.append(parsed['recommendation'])

rec_unique = len(set(recommendations))
print(f'\nRecommendation 唯一數：{rec_unique} / {len(recommendations)}')

# --- 檢查 4：類別分佈 ---
event_types = []
for item in data:
    parsed = json.loads(item['conversations'][1]['value'])
    event_types.append(parsed['event_type'])

print(f'\n類別分佈：')
for et, count in sorted(Counter(event_types).items(), key=lambda x: -x[1]):
    print(f'  {et:8s} → {count} 筆')

# --- 檢查 5：Severity 分佈 ---
severities = [json.loads(item['conversations'][1]['value'])['severity'] for item in data]
print(f'\n嚴重度分佈：')
for sev, count in Counter(severities).most_common():
    print(f'  {sev:10s} → {count} 筆')

Description 唯一數：972 / 1130
Description 唯一率：86.0%
✅ 通過（>= 80%）

Recommendation 唯一數：67 / 1130

類別分佈：
  normal   → 500 筆
  dos      → 500 筆
  probe    → 100 筆
  r2l      → 20 筆
  u2r      → 10 筆

嚴重度分佈：
  high       → 520 筆
  info       → 500 筆
  medium     → 100 筆
  critical   → 10 筆


In [9]:
# --- 範例展示：每類各印 1 筆完整範例 ---
print('=' * 80)
print('每類攻擊的完整範例（供截圖與品質檢查）')
print('=' * 80)

shown_types = set()
for item in data:
    parsed = json.loads(item['conversations'][1]['value'])
    et = parsed['event_type']
    if et not in shown_types:
        shown_types.add(et)
        print(f'\n--- {et.upper()} ---')
        print(f'【Human 輸入】')
        human_text = item['conversations'][0]['value']
        # 只印特徵部分（最後一行）
        lines = human_text.split('\n')
        print(f'  {lines[-1][:120]}...' if len(lines[-1]) > 120 else f'  {lines[-1]}')
        print(f'\n【GPT 輸出】')
        print(json.dumps(parsed, ensure_ascii=False, indent=2))

    if len(shown_types) == 5:
        break

每類攻擊的完整範例（供截圖與品質檢查）

--- NORMAL ---
【Human 輸入】
  duration=0, protocol_type=udp, service=private, flag=SF, src_bytes=105, dst_bytes=0, count=1, srv_count=1, serror_rate=0...

【GPT 輸出】
{
  "event_type": "normal",
  "severity": "info",
  "source_ip": "192.168.1.2",
  "destination_ip": "35.13.209.154",
  "protocol": "udp",
  "description": "一筆正常的 UDP 連線，目標服務為 private，連線旗標 SF，傳送 105 bytes、接收 0 bytes，持續 0 秒。過去 2 秒內同主機連線數=1，同服務連線數=1，目的主機連線數=255，同服務比率=1.0。未偵測到異常行為。",
  "recommendation": "無需處置，此為正常網路連線。持續監控即可。"
}

--- DOS ---
【Human 輸入】
  duration=0, protocol_type=tcp, service=private, flag=S0, src_bytes=0, dst_bytes=0, count=136, srv_count=15, serror_rate=...

【GPT 輸出】
{
  "event_type": "dos",
  "severity": "high",
  "source_ip": "97.198.38.127",
  "destination_ip": "10.0.0.5",
  "protocol": "tcp",
  "description": "SYN Flood 攻擊（neptune）：大量 TCP SYN 封包湧入目標的 private 服務（flag=S0），未完成三向交握，消耗伺服器連線資源。過去 2 秒內同主機連線數=136，SYN 錯誤率=1.0，目的主機連線數=255，不同服務比率=0.06，目的主機 SYN 錯誤率=1.0。",
  "recommendation": "啟用入侵防禦

---

## 小結

### 完成事項

- ✅ 從 NSL-KDD 資料集分層抽樣 **1,130 筆**（Normal 500 / DoS 500 / Probe 100 / R2L 20 / U2R 10）
- ✅ 設計資安事件分析 JSON Schema（7 個欄位）
- ✅ 依 23 種攻擊子類型產生差異化的 description 與 recommendation
- ✅ 以 **ShareGPT 格式**輸出 `cybersecurity_sft_data.json`
- ✅ 通過品質驗證（JSON 合法性、Schema 完整性、description 唯一率 > 80%、類別分佈正確）

### 輸出檔案

- `cybersecurity_sft_data.json`：1,130 筆 ShareGPT 格式訓練資料

### 下一步（Task 3.2）

1. 將 `cybersecurity_sft_data.json` 複製到 LlamaFactory 的 `data/` 目錄
2. 在 `data/dataset_info.json` 中註冊此資料集
3. 撰寫訓練 YAML 設定檔
4. 在 glows.ai（RTX 4090）上執行 LoRA SFT 訓練

### LlamaFactory 資料集註冊方式（ShareGPT 格式）

在 `LLaMA-Factory/data/dataset_info.json` 中加入：

```json
"cybersecurity_sft": {
  "file_name": "cybersecurity_sft_data.json",
  "formatting": "sharegpt",
  "columns": {
    "messages": "conversations"
  },
  "tags": {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt"
  }
}
```